# Maia Lite 0.3.0 — Base Model Evaluation

**Avaliação reproduzível de um modelo causal pré-treinado (sem instruction tuning).**

Inclui: integridade do checkpoint, geração greedy e amostrada, completações PT/EN/ES, lógica, matemática, continuidade textual, avaliação por log-probabilidade de alternativas, perplexidade em corpus *held-out*, exportação JSONL/CSV e gráficos.

> **Integração necessária:** o notebook de treinamento/definição exata da arquitetura não foi disponibilizado. Portanto, este notebook **não recria nem adivinha a arquitetura**. Você deve indicar o módulo Python do projeto que constrói o modelo e o tokenizer, ou fornecer um adaptador na seção 3. As demais seções são executáveis após a integração. Não use `torch.load` em checkpoints não confiáveis.

**Princípios:** preservar tokenizer e pesos originais; registrar versões e hashes; separar testes exploratórios de métricas científicas; não confundir *next-token prediction* com seguimento de instruções.

In [1]:
%pip -q install torch transformers pandas numpy matplotlib tqdm safetensors

## 1. Configuração e ambiente

Se o checkpoint estiver no Drive, habilite `MOUNT_DRIVE=True`. Defina o caminho do arquivo, o diretório do código de treinamento e os identificadores reais do tokenizer. Execute em runtime GPU quando disponível.

In [10]:
from pathlib import Path
import os, sys, json, csv, math, time, random, hashlib, inspect, platform, importlib, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

MOUNT_DRIVE = False
#CHECKPOINT_PATH = '/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0/checkpoints/latest.pt'  # ALTERAR
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
PROJECT_ROOT = '/content/maia-lite'  # diretório contendo o módulo de treinamento, ALTERAR
OUTPUT_DIR = '/content/maia_lite_030_eval'
SEED = 20261116
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float32  # padrão seguro; pode ajustar após validar arquitetura
MAX_CONTEXT = 2048
MAX_PROMPT_TOKENS = 1536

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE, '| torch:', torch.__version__)
print('Checkpoint:', CHECKPOINT_PATH)
print('Projeto:', PROJECT_ROOT)

Device: cuda | torch: 2.14.1+cu130
Checkpoint: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt
Projeto: /content/maia-lite


In [4]:
from pathlib import Path

root = Path('/content/drive/MyDrive')

print("Procurando checkpoints do Maia Lite...\n")

for path in root.rglob('latest.pt'):
    print(path)
    print(f"Tamanho: {path.stat().st_size / 1024**3:.2f} GiB")
    print()

Procurando checkpoints do Maia Lite...



In [14]:
from google.colab import drive
from pathlib import Path
import os

# Montar o Google Drive
drive.mount('/content/drive')

BASE = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/training_runs/"
    "maia_lite_0_3_0_a100_v3_experimental"
)

print("Drive montado:", os.path.ismount("/content/drive"))
print("MyDrive existe:", Path("/content/drive/MyDrive").exists())
print("Diretório do treinamento existe:", BASE.exists())

print("\nConteúdo do diretório:")

if BASE.exists():
    for p in sorted(BASE.iterdir()):
        print(
            p.name,
            f"{p.stat().st_size / 2**30:.3f} GiB"
            if p.is_file() else "<diretório>"
        )
else:
    print("Diretório não encontrado.")

print("\nCheckpoints:")

for name in ["latest.pt", "previous.pt", "latest.pt.tmp"]:
    p = BASE / name
    print(name, "EXISTE" if p.is_file() else "NÃO ENCONTRADO")

Mounted at /content/drive
Drive montado: True
MyDrive existe: True
Diretório do treinamento existe: True

Conteúdo do diretório:
audit_v3_6 <diretório>
audit_v3_7 <diretório>
latest.pt 5.667 GiB
latest.pt.tmp 5.667 GiB
milestone_backups <diretório>
previous.pt 5.667 GiB

Checkpoints:
latest.pt EXISTE
previous.pt EXISTE
latest.pt.tmp EXISTE


## 2. Inspeção não destrutiva do checkpoint

Inspecione primeiro o arquivo real. O formato dos campos é específico do treinador. A leitura completa com `weights_only=True` pode falhar para checkpoints que serializam objetos Python; nesse caso, não use `weights_only=False` sem confiar na origem e entender o conteúdo.

In [15]:
def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

checkpoint_path = Path(CHECKPOINT_PATH)
assert checkpoint_path.is_file(), f'Checkpoint não encontrado: {checkpoint_path}'
checkpoint_hash = sha256_file(checkpoint_path)
print('Bytes:', checkpoint_path.stat().st_size)
print('SHA256:', checkpoint_hash)
try:
    ckpt_preview = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if isinstance(ckpt_preview, dict):
        print('Chaves de nível superior:', list(ckpt_preview)[:50])
        for k,v in ckpt_preview.items():
            if isinstance(v, dict):
                print(k, 'dict', len(v), 'subchaves; primeiras:', list(v)[:8])
            elif isinstance(v, (int,float,str,bool,type(None))): print(k, repr(v))
            else: print(k, type(v).__name__)
    else: print('Objeto:', type(ckpt_preview))
except Exception as e:
    ckpt_preview = None
    print('Leitura segura não suportada pelo formato:', type(e).__name__, str(e)[:300])

Bytes: 6084855785
SHA256: 332be4064daa108588488bb54cefb95ec69233d92e027f0a81e2e3500b380d1d
Leitura segura não suportada pelo formato: UnpicklingError Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.l


In [17]:
import torch

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint carregado!")
print("Tipo:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nChaves disponíveis:")
    for key in checkpoint.keys():
        print(" -", key)

Checkpoint carregado!
Tipo: <class 'dict'>

Chaves disponíveis:
 - identity
 - step
 - model
 - optimizer
 - python_rng
 - numpy_rng
 - torch_rng
 - cuda_rng


In [18]:
import torch

print("Step salvo:", checkpoint["step"])
print("Identity:", checkpoint["identity"])

state = checkpoint["model"]

print("\nTipo dos pesos:", type(state))
print("Número de tensores:", len(state))

print("\nPrimeiras chaves:")
for name in list(state.keys())[:12]:
    print(name, tuple(state[name].shape))

print("\nTotal de parâmetros no checkpoint:")
total = sum(t.numel() for t in state.values())
print(f"{total:,}")

Step salvo: 300000
Identity: {'run_id': 'maia_lite_0_3_0_a100_v3_experimental', 'config': {'vocab_size': 50257, 'block_size': 2048, 'n_layer': 36, 'n_head': 16, 'n_embd': 1024, 'dropout': 0.0, 'bias': True}, 'params': 507026432, 'data_policy': 'experimental_unreviewed', 'approval_sha256': None, 'dataset_blocks': 3607119, 'sources': {'fineweb2_pt': 0.3, 'wikipedia_pt': 0.1, 'fineweb_edu_en': 0.25, 'wikipedia_en': 0.1, 'fineweb2_es': 0.2, 'wikipedia_es': 0.05}, 'microbatch': 2, 'accum': 8, 'seq': 2047, 'seed': 20261105, 'lr': 0.0003, 'min_lr': 3e-05, 'warmup': 2000, 'weight_decay': 0.1, 'grad_clip': 1.0, 'target_updates': 309616, 'torch_version': '2.14.1+cu130'}

Tipo dos pesos: <class 'collections.OrderedDict'>
Número de tensores: 437

Primeiras chaves:
tok.weight (50257, 1024)
pos.weight (2048, 1024)
blocks.0.ln1.weight (1024,)
blocks.0.ln1.bias (1024,)
blocks.0.qkv.weight (3072, 1024)
blocks.0.qkv.bias (3072,)
blocks.0.proj.weight (1024, 1024)
blocks.0.proj.bias (1024,)
blocks.0.ln2.w

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 36
    n_head: int = 16
    n_embd: int = 1024
    dropout: float = 0.0
    bias: bool = True


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c.n_embd

        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3*d, bias=c.bias)
        self.proj = nn.Linear(d, d, bias=c.bias)

        self.ln2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4*d, bias=c.bias)
        self.out = nn.Linear(4*d, d, bias=c.bias)

        self.nh = c.n_head

    def forward(self, x):
        b, t, d = x.shape

        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)

        q = q.view(b, t, self.nh, d//self.nh).transpose(1, 2)
        k = k.view(b, t, self.nh, d//self.nh).transpose(1, 2)
        v = v.view(b, t, self.nh, d//self.nh).transpose(1, 2)

        a = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=0.0
        )

        x = x + self.proj(
            a.transpose(1, 2).contiguous().view(b, t, d)
        )

        x = x + self.out(
            F.gelu(self.fc(self.ln2(x)), approximate="tanh")
        )

        return x


class MaiaGPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c

        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)

        self.blocks = nn.ModuleList([
            Block(c) for _ in range(c.n_layer)
        ])

        self.norm = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(
            c.n_embd, c.vocab_size, bias=False
        )

        # Weight tying
        self.head.weight = self.tok.weight

        self.apply(self.init_weights)

    @staticmethod
    def init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        b, t = idx.shape
        assert t <= self.cfg.block_size

        x = self.tok(idx) + self.pos(
            torch.arange(t, device=idx.device)
        )

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        logits = self.head(x)

        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            targets.reshape(-1),
            ignore_index=-100
        )

        return logits, loss

In [21]:
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

model = MaiaGPT(cfg)

result = model.load_state_dict(
    checkpoint["model"],
    strict=True
)

# Liberar o checkpoint da RAM depois do carregamento
step = checkpoint["step"]
del checkpoint
gc.collect()

model = model.to(device)
model.eval()

print("Maia Lite 0.3.0 carregado!")
print("Step:", step)
print("Device:", device)
print("Parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Compatibilidade:", result)

Maia Lite 0.3.0 carregado!
Step: 300000
Device: cuda
Parâmetros: 507,026,432
Compatibilidade: <All keys matched successfully>


In [23]:
from pathlib import Path
from transformers import AutoTokenizer

TOKENIZER_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer"
)

assert TOKENIZER_DIR.is_dir(), (
    f"Tokenizer não encontrado: {TOKENIZER_DIR}"
)

tokenizer = AutoTokenizer.from_pretrained(
    str(TOKENIZER_DIR),
    local_files_only=True,
    trust_remote_code=False
)

print("Tokenizer carregado!")
print("Vocabulário:", len(tokenizer))

# Teste de codificação e decodificação
texto = "A capital do Brasil é"
ids = tokenizer.encode(texto, add_special_tokens=False)

print("IDs:", ids)
print("Decodificação:", tokenizer.decode(ids))

Tokenizer carregado!
Vocabulário: 50257
IDs: [37, 2517, 402, 3334, 518]
Decodificação: A capital do Brasil é


In [24]:
import torch

assert "model" in globals(), "Modelo não carregado"
assert "tokenizer" in globals(), "Tokenizer não carregado"

prompt = "A capital do Brasil é"

ids = tokenizer.encode(prompt, add_special_tokens=False)
x = torch.tensor([ids], dtype=torch.long, device=device)

model.eval()

with torch.inference_mode():
    for _ in range(30):
        logits, _ = model(x[:, -cfg.block_size:])
        next_token = logits[:, -1, :].argmax(
            dim=-1,
            keepdim=True
        )
        x = torch.cat([x, next_token], dim=1)

print("\nPROMPT:", prompt)
print(
    "RESULTADO:",
    tokenizer.decode(x[0].tolist(), skip_special_tokens=True)
)


PROMPT: A capital do Brasil é
RESULTADO: A capital do Brasil é a capital do Brasil.

A cidade de São Paulo é a capital do Brasil.

História 

A cidade de São Paulo foi fundada


In [25]:
import torch
import torch.nn.functional as F

def generate(
    prompt,
    max_new_tokens=60,
    temperature=0.8,
    top_k=40,
    seed=42
):
    torch.manual_seed(seed)

    ids = tokenizer.encode(
        prompt,
        add_special_tokens=False
    )

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    model.eval()

    with torch.inference_mode():
        for _ in range(max_new_tokens):

            logits, _ = model(
                x[:, -cfg.block_size:]
            )

            logits = logits[:, -1, :].float()
            logits = logits / temperature

            if top_k is not None:
                values, _ = torch.topk(logits, top_k)
                logits[logits < values[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            x = torch.cat([x, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(
        x[0].tolist(),
        skip_special_tokens=True
    )


prompts = [
    "A capital do Brasil é",
    "Brasília é",
    "A inteligência artificial é",
    "The capital of France is",
    "La capital de Argentina es",
    "A soma de dois números inteiros é"
]

for prompt in prompts:
    print("\n" + "=" * 65)
    print("PROMPT:", prompt)
    print("-" * 65)
    print(generate(prompt))


PROMPT: A capital do Brasil é
-----------------------------------------------------------------
A capital do Brasil é o terceiro maior estado do país. Está localizada na região Sudeste do país, na Região Sul. A capital mais populosa do Brasil é Brasília, capital da República Federativa do Brasil, tem aproximadamente 8,7 milhões de habitantes, e o seu segundo maior polo turístico, depois de Natal.
A

PROMPT: Brasília é
-----------------------------------------------------------------
Brasília é o terceiro Estado da federação que mais perde com a falência de um dos seus setores. A queda mais expressiva foi de 20% para o Rio.
O resultado também mostra que os brasileiros não se preocupam com o futuro das pessoas. Há 40 anos, quando o país tinha uma economia

PROMPT: A inteligência artificial é
-----------------------------------------------------------------
A inteligência artificial é o que define as marcas. E quando a questão é o consumidor, a questão é sobre a sua experiência. Por esse

In [26]:
import torch
import torch.nn.functional as F

def score_completion(prefix, completion):
    prefix_ids = tokenizer.encode(
        prefix, add_special_tokens=False
    )
    full_ids = tokenizer.encode(
        prefix + completion, add_special_tokens=False
    )

    assert full_ids[:len(prefix_ids)] == prefix_ids, (
        "A tokenização mudou na fronteira do prefixo."
    )

    ids = torch.tensor(
        [full_ids],
        dtype=torch.long,
        device=device
    )

    with torch.inference_mode():
        logits, _ = model(ids[:, :-1])

    log_probs = F.log_softmax(
        logits.float(), dim=-1
    )

    positions = torch.arange(
        len(prefix_ids) - 1,
        len(full_ids) - 1,
        device=device
    )

    targets = ids[0, len(prefix_ids):]

    selected = log_probs[0, positions, targets]

    return selected.sum().item(), selected.mean().item()


tests = [
    (
        "A capital do Brasil é",
        [" Brasília.", " São Paulo.", " Rio de Janeiro."]
    ),
    (
        "The capital of France is",
        [" Paris.", " London.", " Berlin."]
    ),
    (
        "La capital de Argentina es",
        [" Buenos Aires.", " Córdoba.", " Rosario."]
    ),
    (
        "2 + 2 =",
        [" 4", " 5", " 3"]
    )
]

for prefix, completions in tests:
    print("\n" + "=" * 65)
    print(prefix)

    results = []

    for completion in completions:
        total, mean = score_completion(prefix, completion)
        results.append((completion, total, mean))

    for completion, total, mean in sorted(
        results, key=lambda x: x[1], reverse=True
    ):
        print(
            f"{completion:22s} "
            f"logP={total:9.3f} "
            f"mean={mean:8.3f}"
        )


A capital do Brasil é
 São Paulo.            logP=   -5.556 mean=  -1.852
 Brasília.             logP=   -6.840 mean=  -3.420
 Rio de Janeiro.       logP=   -7.480 mean=  -1.870

The capital of France is
 Paris.                logP=   -2.289 mean=  -1.144
 London.               logP=   -6.789 mean=  -3.395
 Berlin.               logP=   -8.409 mean=  -4.205

La capital de Argentina es
 Buenos Aires.         logP=   -5.877 mean=  -1.959
 Córdoba.              logP=   -9.013 mean=  -4.506
 Rosario.              logP=  -11.322 mean=  -5.661

2 + 2 =
 4                     logP=   -2.599 mean=  -2.599
 3                     logP=   -2.937 mean=  -2.937
 5                     logP=   -3.254 mean=  -3.254


In [27]:
tests = [
    (
        "A capital do Brasil é",
        [" Brasília.", " São Paulo."]
    ),
    (
        "Brasília é a capital",
        [" do Brasil.", " de Portugal."]
    ),
    (
        "O Brasil tem como capital",
        [" Brasília.", " São Paulo."]
    ),
    (
        "Brazil's capital is",
        [" Brasília.", " São Paulo."]
    ),
    (
        "La capital de Brasil es",
        [" Brasilia.", " São Paulo."]
    ),
    (
        "A capital da França é",
        [" Paris.", " Londres."]
    ),
    (
        "The capital of France is",
        [" Paris.", " London."]
    ),
]

for prefix, completions in tests:
    print("\n", prefix)

    results = []

    for completion in completions:
        total, mean = score_completion(prefix, completion)
        results.append((completion, total, mean))

    results.sort(key=lambda x: x[1], reverse=True)

    for completion, total, mean in results:
        print(
            f"{completion:20s} "
            f"logP={total:8.3f} "
            f"mean={mean:8.3f}"
        )


 A capital do Brasil é
 São Paulo.          logP=  -5.556 mean=  -1.852
 Brasília.           logP=  -6.840 mean=  -3.420

 Brasília é a capital
 do Brasil.          logP=  -2.646 mean=  -0.882
 de Portugal.        logP=  -8.651 mean=  -2.884

 O Brasil tem como capital
 Brasília.           logP=  -3.306 mean=  -1.653
 São Paulo.          logP=  -4.536 mean=  -1.512

 Brazil's capital is
 São Paulo.          logP=  -5.238 mean=  -1.746
 Brasília.           logP=  -5.397 mean=  -2.699

 La capital de Brasil es
 Brasilia.           logP=  -6.935 mean=  -2.312
 São Paulo.          logP=  -6.961 mean=  -2.320

 A capital da França é
 Paris.              logP=  -2.774 mean=  -1.387
 Londres.            logP=  -7.354 mean=  -3.677

 The capital of France is
 Paris.              logP=  -2.289 mean=  -1.144
 London.             logP=  -6.789 mean=  -3.395


In [28]:
import torch
import torch.nn.functional as F

def compare_next_token(prefix, alternatives):
    ids = tokenizer.encode(
        prefix,
        add_special_tokens=False
    )

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    with torch.inference_mode():
        logits, _ = model(x)

    log_probs = F.log_softmax(
        logits[0, -1].float(),
        dim=-1
    )

    results = []

    for alternative in alternatives:
        full = tokenizer.encode(
            prefix + alternative,
            add_special_tokens=False
        )

        if full[:len(ids)] != ids:
            print(
                "Tokenização incompatível:",
                repr(alternative)
            )
            continue

        continuation = full[len(ids):]

        if not continuation:
            continue

        token_id = continuation[0]

        results.append({
            "alternative": alternative,
            "first_token": tokenizer.decode([token_id]),
            "logP": log_probs[token_id].item(),
            "prob": log_probs[token_id].exp().item()
        })

    for result in sorted(
        results,
        key=lambda r: r["logP"],
        reverse=True
    ):
        print(
            f"{result['alternative']:20s} "
            f"token={result['first_token']!r:14s} "
            f"logP={result['logP']:8.3f} "
            f"P={result['prob']:.4%}"
        )


compare_next_token(
    "A capital do Brasil é",
    [" Brasília.", " São Paulo.", " Rio de Janeiro."]
)

 São Paulo.          token=' São'         logP=  -4.313 P=1.3397%
 Brasília.           token=' Brasília'    logP=  -5.732 P=0.3241%
 Rio de Janeiro.     token=' Rio'         logP=  -6.257 P=0.1917%


In [29]:
import torch
import torch.nn.functional as F

prompt = "A capital do Brasil é"

ids = tokenizer.encode(
    prompt,
    add_special_tokens=False
)

x = torch.tensor(
    [ids],
    dtype=torch.long,
    device=device
)

model.eval()

with torch.inference_mode():
    logits, _ = model(x)

probs = F.softmax(
    logits[0, -1].float(),
    dim=-1
)

values, indices = torch.topk(probs, 20)

print("PROMPT:", prompt)
print("\nTOP 20 PRÓXIMOS TOKENS\n")

for rank, (token_id, prob) in enumerate(
    zip(indices.tolist(), values.tolist()), 1
):
    token = tokenizer.decode([token_id])
    print(
        f"{rank:2d}. {token!r:22s} "
        f"{prob * 100:8.4f}%"
    )

PROMPT: A capital do Brasil é

TOP 20 PRÓXIMOS TOKENS

 1. ' a'                    29.7670%
 2. ' uma'                  13.9681%
 3. ' o'                     7.6300%
 4. ' um'                    4.4282%
 5. ' considerada'           4.0304%
 6. ' também'                1.9391%
 7. ','                      1.8306%
 8. ' conhecida'             1.3897%
 9. ' São'                   1.3397%
10. ' hoje'                  0.7481%
11. ' ainda'                 0.6953%
12. ' mais'                  0.6509%
13. ' muito'                 0.6312%
14. ' sua'                   0.5822%
15. ' B'                     0.5760%
16. ' de'                    0.4788%
17. ' sede'                  0.4453%
18. ' dividida'              0.4107%
19. ' Porto'                 0.4042%
20. ' formada'               0.3900%


In [30]:
prompts = [
    "A capital do Brasil é",

    "O Brasil é um país da América do Sul. Sua capital é",

    "Brasília é uma cidade brasileira. Ela é a capital do",

    "Pergunta: Qual é a capital do Brasil?\nResposta:",

    "Brasil — capital:",

    "O nome da capital brasileira é"
]

for prompt in prompts:
    print("\n" + "=" * 70)
    print("PROMPT:", repr(prompt))
    print("-" * 70)

    print(
        generate(
            prompt,
            max_new_tokens=35,
            temperature=0.7,
            top_k=30,
            seed=42
        )
    )


PROMPT: 'A capital do Brasil é'
----------------------------------------------------------------------
A capital do Brasil é o centro político-administrativo da nação.

Distritos 
O Brasil possui oito distritos. São eles:

 Belo Horizonte:
 Alto do

PROMPT: 'O Brasil é um país da América do Sul. Sua capital é'
----------------------------------------------------------------------
O Brasil é um país da América do Sul. Sua capital é o Rio de Janeiro. A sua população é de aproximadamente 3,4 milhões de pessoas, que são o que chamamos de um quarto da população mundial.

A língua oficial

PROMPT: 'Brasília é uma cidade brasileira. Ela é a capital do'
----------------------------------------------------------------------
Brasília é uma cidade brasileira. Ela é a capital do estado do Paraná e da Região Norte do Brasil, localizada na região Norte, na Região Sul do Brasil.

História 
Por volta de 1390, o então governador

PROMPT: 'Pergunta: Qual é a capital do Brasil?\nResposta:'
-------------

## 3. Adaptador obrigatório da arquitetura e do tokenizer

**Escolha uma das opções:**

- **A (recomendada):** no código do projeto, exponha `load_maia_for_eval(checkpoint_path, device)` que retorne `(model, tokenizer, metadata)`. O modelo deve aceitar `input_ids` e retornar logits `[batch, seq, vocab]` (diretamente, em `output.logits`, ou como primeiro elemento). O tokenizer deve implementar `encode(text)` e `decode(ids)`.
- **B:** implemente `load_maia_for_eval` diretamente na célula seguinte, usando **exatamente** as classes/configurações originais, sem alterar o vocabulário nem a arquitetura.

**Atenção:** a identificação de `state_dict` não permite reconstruir automaticamente um Transformer. O adaptador deve restaurar pesos e configuração de maneira estrita, preferencialmente com `strict=True`, e selecionar pesos do modelo, não do otimizador.

## 4. Validação da interface, tokenizer e forward

Os testes não iniciam se o tokenizer ou os logits estiverem inconsistentes. Para modelos com assinatura especial de forward, adapte `forward_logits` sem mudar o modelo.

## 5. Motor de geração

O código abaixo implementa geração autoregressiva simples e transparente. A geração greedy não usa temperatura. A amostragem usa gerador com semente definida por teste. Não há *chat template*, penalidades de repetição ou pós-processamento oculto. Se o seu modelo exige máscara causal ou positional IDs explícitos, configure-os em `forward_logits`.

## 6. Bateria PT / EN / ES

Prompts de *completação*, não instruções de chat. Para padrões e matemática, utilizamos também contexto exemplificado (*few-shot*). As respostas livres não recebem automaticamente nota de acerto; ficam disponíveis para inspeção humana.

In [32]:
TESTS = [
 ('pt_capital','pt','factual','A capital do Brasil é'),
 ('pt_water','pt','factual','A água congela, ao nível do mar, a aproximadamente'),
 ('pt_science','pt','continuation','A inteligência artificial é uma área da ciência da computação que'),
 ('pt_climate','pt','continuation','As mudanças climáticas representam um desafio porque'),
 ('pt_seq','pt','pattern','Sequência: 2, 4, 6, 8, 10,'),
 ('pt_logic','pt','logic','Todos os cães são mamíferos. Todos os mamíferos são animais. Portanto, todos os cães são'),
 ('pt_math','pt','math','2 + 3 = 5\n4 + 5 = 9\n7 + 2 ='),
 ('pt_reason','pt','logic','Ana é mais alta que Bruno. Bruno é mais alto que Carlos. Portanto, Ana é'),
 ('en_capital','en','factual','The capital of France is'),
 ('en_science','en','continuation','Machine learning is a field of artificial intelligence that'),
 ('en_seq','en','pattern','Sequence: 1, 2, 4, 8, 16,'),
 ('en_logic','en','logic','All dogs are mammals. All mammals are animals. Therefore, all dogs are'),
 ('en_math','en','math','2 + 3 = 5\n4 + 5 = 9\n7 + 2 ='),
 ('es_capital','es','factual','La capital de España es'),
 ('es_science','es','continuation','La fotosíntesis es el proceso por el cual'),
 ('es_seq','es','pattern','Secuencia: 3, 6, 9, 12,'),
 ('es_logic','es','logic','Todos los perros son mamíferos. Todos los mamíferos son animales. Por lo tanto, todos los perros son'),
 ('es_math','es','math','2 + 3 = 5\n4 + 5 = 9\n7 + 2 ='),
]
TESTS = [{'id':a,'language':b,'category':c,'prompt':d} for a,b,c,d in TESTS]
CONFIGS = [
 {'name':'greedy','strategy':'greedy','temperature':1.0,'top_p':1.0},
 {'name':'conservative','strategy':'sample','temperature':0.3,'top_p':0.9},
 {'name':'exploratory','strategy':'sample','temperature':0.8,'top_p':0.95},
]
print(f'{len(TESTS)} testes x {len(CONFIGS)} configurações = {len(TESTS)*len(CONFIGS)} gerações')

18 testes x 3 configurações = 54 gerações


In [33]:
MAX_NEW_TOKENS_BY_CATEGORY = {'factual':32,'continuation':96,'pattern':32,'logic':48,'math':32}
results=[]
for i,test in enumerate(tqdm(TESTS,desc='Bateria')):
    for j,cfg in enumerate(CONFIGS):
        generated=generate(test['prompt'], strategy=cfg['strategy'],
            temperature=cfg['temperature'],top_p=cfg['top_p'],
            max_new_tokens=MAX_NEW_TOKENS_BY_CATEGORY[test['category']],
            seed=SEED+i*100+j)
        results.append({**test,**cfg,**generated,'seed':SEED+i*100+j,
                        'checkpoint_sha256':checkpoint_hash})
results_df=pd.DataFrame(results)
display(results_df[['id','language','category','name','prompt','continuation']])

Bateria:   0%|          | 0/18 [00:00<?, ?it/s]

TypeError: MaiaGPT.forward() got an unexpected keyword argument 'input_ids'

## 7. Testes objetivos por probabilidade condicional

Uma avaliação de *base model* mais estável do que a resposta livre é perguntar: **qual alternativa o modelo considera mais provável como continuação do prefixo?**

A pontuação padrão usa log-probabilidade **média por token** para reduzir o viés de comprimento, mas essa normalização também pode introduzir vieses. Incluímos a soma de log-probabilidades para auditoria. As alternativas devem ser cuidadosamente balanceadas e a tokenização da junção `prefixo + alternativa` precisa preservar o prefixo como sequência de tokens. Se não preservar, o teste é marcado como incompatível (não pontuado).

In [34]:
CHOICES = [
 {'id':'pt_capital_choice','language':'pt','category':'factual','prefix':'A capital do Brasil é',
  'choices':[' Brasília.',' Buenos Aires.',' Lisboa.',' Santiago.'],'correct':0},
 {'id':'en_capital_choice','language':'en','category':'factual','prefix':'The capital of France is',
  'choices':[' Paris.',' Rome.',' Madrid.',' Berlin.'],'correct':0},
 {'id':'es_capital_choice','language':'es','category':'factual','prefix':'La capital de España es',
  'choices':[' Madrid.',' Lisboa.',' Roma.',' París.'],'correct':0},
 {'id':'pt_logic_choice','language':'pt','category':'logic',
  'prefix':'Todos os cães são mamíferos. Todos os mamíferos são animais. Portanto, todos os cães são',
  'choices':[' animais.',' répteis.',' plantas.',' minerais.'],'correct':0},
 {'id':'en_logic_choice','language':'en','category':'logic',
  'prefix':'All dogs are mammals. All mammals are animals. Therefore, all dogs are',
  'choices':[' animals.',' reptiles.',' plants.',' minerals.'],'correct':0},
 {'id':'es_logic_choice','language':'es','category':'logic',
  'prefix':'Todos los perros son mamíferos. Todos los mamíferos son animales. Por lo tanto, todos los perros son',
  'choices':[' animales.',' reptiles.',' plantas.',' minerales.'],'correct':0},
 {'id':'pt_math_choice','language':'pt','category':'math','prefix':'2 + 3 = 5\n4 + 5 = 9\n7 + 2 =',
  'choices':[' 9',' 8',' 10',' 11'],'correct':0},
 {'id':'en_pattern_choice','language':'en','category':'pattern','prefix':'Sequence: 1, 2, 4, 8, 16,',
  'choices':[' 32',' 18',' 24',' 30'],'correct':0},
 {'id':'es_pattern_choice','language':'es','category':'pattern','prefix':'Secuencia: 3, 6, 9, 12,',
  'choices':[' 15',' 16',' 18',' 24'],'correct':0},
]

def score_continuation(prefix, suffix):
    prefix_ids=encode(prefix)
    full_ids=encode(prefix+suffix)
    if full_ids[:len(prefix_ids)] != prefix_ids:
        raise ValueError('A tokenização mudou na fronteira prefixo/sufixo; revise o template')
    if len(full_ids)<=len(prefix_ids): raise ValueError('Sufixo sem tokens')
    if len(full_ids)>MAX_CONTEXT: raise ValueError('Contexto excedido')
    # Logit da posição t-1 prevê token na posição t.
    if len(prefix_ids)<1: raise ValueError('Prefixo vazio')
    with torch.inference_mode():
        x=torch.tensor([full_ids],device=DEVICE,dtype=torch.long)
        logits=forward_logits(x)[0]
        log_probs=torch.log_softmax(logits.float(),dim=-1)
        positions=torch.arange(len(prefix_ids)-1,len(full_ids)-1,device=DEVICE)
        targets=torch.tensor(full_ids[len(prefix_ids):],device=DEVICE)
        token_logps=log_probs[positions,targets]
    return {'mean_logprob':float(token_logps.mean()),
            'sum_logprob':float(token_logps.sum()),'num_tokens':len(targets)}

choice_rows=[]
for item in tqdm(CHOICES,desc='Escolha objetiva'):
    try:
        scored=[score_continuation(item['prefix'],ch) for ch in item['choices']]
        pred=int(np.argmax([s['mean_logprob'] for s in scored]))
        choice_rows.append({'id':item['id'],'language':item['language'],'category':item['category'],
          'prediction':pred,'correct':item['correct'],'hit':int(pred==item['correct']),
          'prediction_text':item['choices'][pred], 'scores':scored,'status':'ok'})
    except Exception as e:
        choice_rows.append({'id':item['id'],'language':item['language'],'category':item['category'],
          'prediction':None,'correct':item['correct'],'hit':None,
          'prediction_text':None,'scores':None,'status':f'ERROR: {e}'})
choice_df=pd.DataFrame(choice_rows)
display(choice_df[['id','language','category','prediction_text','hit','status']])
valid=choice_df[choice_df['status']=='ok']
if len(valid):
    print('Acurácia total:',round(valid['hit'].mean(),3),'N=',len(valid))
    display(valid.groupby(['language','category'])['hit'].agg(['mean','count']))
else: print('Nenhum teste objetivo válido; revise tokenização e prompts')

Escolha objetiva:   0%|          | 0/9 [00:00<?, ?it/s]

,id,language,category,prediction_text,hit,status
0,pt_capital_choice,pt,factual,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
1,en_capital_choice,en,factual,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
2,es_capital_choice,es,factual,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
3,pt_logic_choice,pt,logic,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
4,en_logic_choice,en,logic,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
5,es_logic_choice,es,logic,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
6,pt_math_choice,pt,math,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
7,en_pattern_choice,en,pattern,None,None,ERROR: MaiaGPT.forward() got an unexpected key...
8,es_pattern_choice,es,pattern,None,None,ERROR: MaiaGPT.forward() got an unexpected key...


Nenhum teste objetivo válido; revise tokenização e prompts


## 8. Perplexidade em dados independentes

**Obrigatório fornecer um corpus realmente independente do treinamento**, idealmente um arquivo por idioma, sem duplicatas nem trechos memorizados. O exemplo não utiliza texto de treinamento como validação. A métrica abaixo calcula a NLL token a token, em janelas **não sobrepostas** de até `MAX_CONTEXT` tokens. A primeira posição de cada janela não é pontuada, pois não tem contexto anterior dentro dela. Para uma avaliação de referência, mantenha esse protocolo idêntico entre checkpoints.

Se o arquivo for muito longo, limite o número de tokens por idioma de forma documentada. Não interprete a perplexidade sem verificar a compatibilidade do tokenizer.

In [35]:
HELDOUT_FILES = {
 'pt': '/content/heldout_pt.txt',
 'en': '/content/heldout_en.txt',
 'es': '/content/heldout_es.txt',
}
MAX_EVAL_TOKENS_PER_LANGUAGE = 200_000
EVAL_WINDOW = min(MAX_CONTEXT,2048)

def perplexity_from_text(text, max_tokens=MAX_EVAL_TOKENS_PER_LANGUAGE):
    token_ids=encode(text)[:max_tokens]
    if len(token_ids)<2: raise ValueError('São necessários pelo menos dois tokens')
    total_nll=0.0; total_count=0
    for start in tqdm(range(0,len(token_ids),EVAL_WINDOW),leave=False):
        chunk=token_ids[start:start+EVAL_WINDOW]
        if len(chunk)<2: continue
        x=torch.tensor([chunk],device=DEVICE,dtype=torch.long)
        with torch.inference_mode():
            logits=forward_logits(x)[:,:-1,:].float()
            labels=x[:,1:]
            nll=torch.nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),
                         labels.reshape(-1),reduction='sum')
        total_nll+=float(nll.item()); total_count+=labels.numel()
    if not total_count: raise ValueError('Nenhum token pontuado')
    mean_nll=total_nll/total_count
    return {'nll':mean_nll,'perplexity':math.exp(mean_nll),
            'scored_tokens':total_count,'available_tokens':len(token_ids)}

ppl_rows=[]
for lang,path in HELDOUT_FILES.items():
    p=Path(path)
    if not p.exists():
        print(f'{lang}: arquivo não encontrado ({p}); pulando, sem inventar métrica')
        continue
    text=p.read_text(encoding='utf-8')
    out=perplexity_from_text(text)
    ppl_rows.append({'language':lang,'file':str(p),'sha256':sha256_file(p),**out})
ppl_df=pd.DataFrame(ppl_rows)
if len(ppl_df): display(ppl_df)
else: print('Forneça arquivos held-out PT/EN/ES para obter perplexidade')

pt: arquivo não encontrado (/content/heldout_pt.txt); pulando, sem inventar métrica
en: arquivo não encontrado (/content/heldout_en.txt); pulando, sem inventar métrica
es: arquivo não encontrado (/content/heldout_es.txt); pulando, sem inventar métrica
Forneça arquivos held-out PT/EN/ES para obter perplexidade


## 9. Inspeção humana e gráficos

As completações livres precisam de leitura crítica. A planilha inclui campos vazios para atribuir notas humanas de 0 a 3 para coerência, correção e repetição. Evite transformar automaticamente avaliações subjetivas em acurácia.

In [36]:
review_df=results_df.copy()
for col in ['coherence_0_3','factuality_0_3','repetition_0_3','reviewer_notes']:
    review_df[col]=''
print('Amostra para revisão:')
display(review_df[['id','name','prompt','continuation','coherence_0_3','factuality_0_3']].head(12))
if len(valid):
    fig,ax=plt.subplots(figsize=(7,4))
    valid.groupby('language')['hit'].mean().plot(kind='bar',ax=ax)
    ax.set_ylim(0,1);ax.set_ylabel('Acurácia');ax.set_title('Testes objetivos por idioma (amostra pequena)')
    plt.tight_layout();plt.show()
if len(ppl_df):
    fig,ax=plt.subplots(figsize=(7,4))
    ax.bar(ppl_df['language'],ppl_df['perplexity'])
    ax.set_ylabel('Perplexidade');ax.set_title('Corpus held-out por idioma')
    plt.tight_layout();plt.show()

NameError: name 'results_df' is not defined

## 10. Exportação e manifesto de reprodutibilidade

Salva resultados e parâmetros. O hash do checkpoint permite identificar exatamente os pesos avaliados. A execução pode ser comparada com checkpoints posteriores, desde que corpus, tokenizer, prompts, geração e protocolo sejam preservados.

In [ ]:
out=Path(OUTPUT_DIR); out.mkdir(parents=True,exist_ok=True)
results_df.to_csv(out/'generations.csv',index=False)
review_df.to_csv(out/'human_review_template.csv',index=False)
choice_df.to_csv(out/'objective_choices.csv',index=False)
if len(ppl_df): ppl_df.to_csv(out/'heldout_perplexity.csv',index=False)
with open(out/'generations.jsonl','w',encoding='utf-8') as f:
    for record in results: f.write(json.dumps(record,ensure_ascii=False,default=str)+'\n')
with open(out/'objective_choices.jsonl','w',encoding='utf-8') as f:
    for record in choice_rows: f.write(json.dumps(record,ensure_ascii=False,default=str)+'\n')
manifest={
 'project':'Maia Lite 0.3.0','evaluation':'base_model',
 'checkpoint_path':str(checkpoint_path),'checkpoint_sha256':checkpoint_hash,
 'checkpoint_bytes':checkpoint_path.stat().st_size,
 'adapter_module':ADAPTER_MODULE,'model_metadata':model_metadata,
 'tokenizer_class':type(tokenizer).__name__,'model_class':type(model).__name__,
 'torch_version':torch.__version__,'python_version':platform.python_version(),
 'device':DEVICE,'seed':SEED,'max_context':MAX_CONTEXT,
 'generation_configs':CONFIGS,'num_prompts':len(TESTS),'num_objective_tests':len(CHOICES),
 'heldout_files':HELDOUT_FILES,'heldout_results':ppl_rows,
 'limitations':['architecture supplied externally','small illustrative objective battery',
                'manual review required for free text','heldout data not included'],
}
with open(out/'manifest.json','w',encoding='utf-8') as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2,default=str)
print('Arquivos exportados:')
for p in sorted(out.iterdir()): print(p.name, p.stat().st_size,'bytes')
# Opcional no Colab:
# from google.colab import files
# files.download(str(out/'generations.csv'))

## 11. Critérios de interpretação e próximos experimentos

- **Factual:** testar alternativas com comprimentos e frequências comparáveis; inverter a ordem das opções e variar a formulação do prefixo.
- **Lógica:** incluir problemas sintéticos inéditos para distinguir raciocínio de memorização de frases comuns.
- **Matemática:** separar aritmética, sequências e generalização; comparar formatos zero-shot e few-shot.
- **Idiomas:** usar o mesmo número de exemplos e textos held-out comparáveis em PT, EN e ES.
- **Validação:** evitar vazamento de dados, registrar o hash de cada corpus e comparar NLL/perplexidade no mesmo tokenizer.
- **Checkpoints:** avaliar versões intermediárias e final sob os mesmos prompts, sementes e condições.

**Limitações atuais:** os exemplos de múltipla escolha são um teste de fumaça, não um benchmark estatisticamente representativo. A perplexidade de diferentes tokenizers não é diretamente comparável. O método de NLL usa janelas independentes; documente isso em qualquer relatório científico. Não é possível afirmar que o modelo aprendeu lógica apenas por completar corretamente frases conhecidas.

**Checklist final:** checkpoint íntegro; adaptador com `strict=True`; tokenizer idêntico ao treinamento; resultados reproduzíveis; corpus held-out independente; relatório revisado.